In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch 
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader


In [3]:
transform = v2.Compose([v2.ToImage(),v2.ToDtype(dtype=torch.float32,scale=True)])

training_data = datasets.CIFAR10(
    root="../data",download=True,transform=transform,train=True
)
test_data = datasets.CIFAR10(
    root="../data",download=True,transform=transform,train=False
)

In [4]:
print(len(training_data))
print(len(test_data))

50000
10000


In [5]:
train_loader = DataLoader(training_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False)

In [6]:
images,labels = next(iter(train_loader))
print(images.shape)
len(train_loader)

torch.Size([64, 3, 32, 32])


782

In [29]:
class TwoLayerNeuralNet:
    def __init__(self, input_size, hidden_layer_size, output_layer_size):
        """
        input_size = 3 * 32 * 32 = 3072
        hidden_layer_size = 256
        output_layer_size = 10
        """

        self.input_size = input_size
        self.hidden_layer_size = hidden_layer_size
        self.output_layer_size = output_layer_size

        #weight initialization 
        self.w1 = np.random.rand(hidden_layer_size,input_size)
        self.w2 = np.random.rand(output_layer_size,hidden_layer_size) 
        self.b1 = np.zeros(shape=hidden_layer_size)
        self.b2 = np.zeros(shape=output_layer_size)

    # def sigmoid(self,x): 
    #     return 1/(1+np.exp(-x))

    def ReLU(self,x): 
        return np.maximum(0, x)

    def forward(self, x):
        """
        x shape : (batch_Size,input_size)
        every image in x is a flattened 1d pixel of an image
        
        Compute:

        first affine layer
        ReLU activation
        second affine layer
        class scores

        Store intermediate values required by backward().
        """
        
        #First layer computation
        # (B, 3072) @ (3072, 256) -> (B, 256)
        z1 = np.matmul(x, self.w1.T) + self.b1

        #add non linearity 
         # (B, 256)
        hidden = self.ReLU(z1)

        # Second layer computation
         # (B, 256) @ (256, 10) -> (B, 10)
        scores = np.matmul(hidden,self.w2.T) + self.b2 

        #computing prob 
         # (B, 10)
        y_hat = self.softmax(scores)

        self.cache = (x, z1, hidden,y_hat)#will be needed for backpropogation

        return scores

    def softmax(self,logits):
        """here we pass the 10 scores that we have recieved to convert them into probabilities"""
        """
        logits shape: (batch_size, 10)
        """

        shifted_logits = logits - np.max(
        logits,
        axis=1,
        keepdims=True
        )

        exp_scores = np.exp(shifted_logits)

        probabilities = exp_scores/np.sum(exp_scores,axis=1,keepdims=True)

        return probabilities

        
    def CrossEntropyLoss(self, y, y_hat):
        """
        y shape:     (batch_size,)
        y_hat shape: (batch_size, 10)
        """
        batch_size = y.shape[0]

        correct_class_probabilities = y_hat[
            np.arange(batch_size),
            y
        ]

        loss = -np.mean(
            np.log(correct_class_probabilities + 1e-12)
        )

        return loss

    
    def backward(self, y):
        """
        y shape: (batch_size,)
        """
        
        x, z1, hidden, y_hat = self.cache
        batch_size = x.shape[0]

        # Softmax + cross-entropy gradient
        dz2 = y_hat.copy()
        dz2[np.arange(batch_size), y] -= 1
        dz2 /= batch_size
    
        # Second affine layer
        dw2 = dz2.T @ hidden
        db2 = np.sum(dz2, axis=0)
    
        dhidden = dz2 @ self.w2
    
        # ReLU backward
        dz1 = dhidden * (z1 > 0)
    
        # First affine layer
        dw1 = dz1.T @ x
        db1 = np.sum(dz1, axis=0)


        return dw1, db1, dw2, db2
        

        

        #writting backpropogation needs practice fuuh
        
        
        

In [30]:
model = TwoLayerNeuralNet(3 * 32 * 32, 256, 10)

In [33]:
epochs = 5
learning_rate = 0.01

for epoch in range(epochs):

    for batch_idx, (images, labels) in enumerate(train_loader):

        batch_size = images.shape[0]

        x = images.numpy().reshape(batch_size, -1)
        y = labels.numpy()

        # Forward pass
        model.forward(x)

        # Probabilities from cache
        _, _, _, y_hat = model.cache

        # Average batch loss
        loss = model.CrossEntropyLoss(y, y_hat)

        # Batch accuracy
        predictions = np.argmax(y_hat, axis=1)
        accuracy = np.mean(predictions == y)

        # Backward pass
        dw1, db1, dw2, db2 = model.backward(y)

        # SGD update
        model.w1 -= learning_rate * dw1
        model.b1 -= learning_rate * db1
        model.w2 -= learning_rate * dw2
        model.b2 -= learning_rate * db2

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Batch [{batch_idx + 1}/{len(train_loader)}] "
            f"Loss: {loss:.4f} "
            f"Accuracy: {accuracy * 100:.2f}%"
        )

Epoch [1/5] Batch [1/782] Loss: 2.6462 Accuracy: 21.88%
Epoch [1/5] Batch [2/782] Loss: 2.1832 Accuracy: 26.56%
Epoch [1/5] Batch [3/782] Loss: 2.2703 Accuracy: 32.81%
Epoch [1/5] Batch [4/782] Loss: 2.1210 Accuracy: 35.94%
Epoch [1/5] Batch [5/782] Loss: 1.8896 Accuracy: 31.25%
Epoch [1/5] Batch [6/782] Loss: 1.8431 Accuracy: 32.81%
Epoch [1/5] Batch [7/782] Loss: 1.7632 Accuracy: 35.94%
Epoch [1/5] Batch [8/782] Loss: 2.0318 Accuracy: 26.56%
Epoch [1/5] Batch [9/782] Loss: 1.9825 Accuracy: 28.12%
Epoch [1/5] Batch [10/782] Loss: 1.9998 Accuracy: 28.12%
Epoch [1/5] Batch [11/782] Loss: 2.2434 Accuracy: 28.12%
Epoch [1/5] Batch [12/782] Loss: 1.9105 Accuracy: 35.94%
Epoch [1/5] Batch [13/782] Loss: 1.8662 Accuracy: 25.00%
Epoch [1/5] Batch [14/782] Loss: 1.8223 Accuracy: 34.38%
Epoch [1/5] Batch [15/782] Loss: 2.0348 Accuracy: 25.00%
Epoch [1/5] Batch [16/782] Loss: 2.1506 Accuracy: 31.25%
Epoch [1/5] Batch [17/782] Loss: 1.9228 Accuracy: 32.81%
Epoch [1/5] Batch [18/782] Loss: 1.7766 